# Projekt 2 — Klasyfikacja jakości wina (Wine Quality)

**Autor:** Olko Dawid · **Przedmiot:** Sztuczna inteligencja

## Cel notebooka

Ten notebook prowadzi krok po kroku przez:
1. wczytanie i **eksplorację** zbioru `WineQT.csv`,
2. uruchomienie eksperymentu z modułu `src/experiment.py` (MLP + Random Forest + opcjonalnie XGBoost),
3. **wizualizację wyników** walidacji krzyżowej i macierzy pomyłek,
4. **wnioski** merytoryczne pod sprawozdanie.

**Zbiór:** ~6497 próbek wina, **11 cech chemicznych**, etykieta **`quality`** (oceny 3–9) — klasyfikacja **wieloklasowa**, nie binarna.

> Uruchom komórki **od góry do dołu** (`Run All`). Pierwsze uruchomienie eksperymentu może potrwać 1–3 minuty.


## 0. Konfiguracja środowiska

Importujemy biblioteki i ustawiamy ścieżkę do katalogu projektu (tam jest folder `src/`).
Wykresy wyświetlą się **w notebooku** dzięki `%matplotlib inline`.


In [ ]:
%matplotlib inline

import warnings
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

# Katalog projektu (notebook w root lub w docs/)
ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import (
    DATA_PATH,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    ID_COLUMN,
    RESULTS_DIR,
    RANDOM_STATE,
)
from src.experiment import wczytaj_pelna_ramke, podziel_na_cechy_i_etykiete, uruchom_pelny_eksperyment

print("Katalog projektu:", ROOT.resolve())
print("Plik danych:", DATA_PATH.resolve())
print("Istnieje:", DATA_PATH.is_file())


## 1. Wczytanie danych

Funkcja `wczytaj_pelna_ramke`:
- wczytuje CSV,
- sprawdza brakujące wartości,
- weryfikuje obecność 11 cech i kolumny `quality`.

Kolumna **`Id`** to tylko identyfikator wiersza — **nie** trafia do modelu.


In [ ]:
df = wczytaj_pelna_ramke(DATA_PATH)

print(f"Liczba rekordów: {len(df)}")
print(f"Liczba kolumn: {df.shape[1]}")
print(f"Zakres ocen quality: {df[TARGET_COLUMN].min()} – {df[TARGET_COLUMN].max()}")
print()
display(df[FEATURE_COLUMNS + [TARGET_COLUMN]].head(8))


### 1.1 Rozkład klas (`quality`)

Wino ma **nierówny rozkład ocen** — najwięcej próbek z oceną **5** i **6**.  
Przy takim rozkładzie sama **accuracy** może być wysoka, jeśli model „zawsze” przewiduje klasę 5 lub 6. Dlatego w projekcie liczymy też **balanced accuracy** i **F1-macro**.


In [ ]:
vc = df[TARGET_COLUMN].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(vc.index.astype(str), vc.values, color=sns.color_palette("Blues", len(vc)))
ax.set_xlabel("Ocena jakości (quality)")
ax.set_ylabel("Liczba próbek")
ax.set_title("Rozkład klas — Wine Quality")
for b, v in zip(bars, vc.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 30, str(v), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print(vc)
print(f"\nNajczęstsza klasa: {vc.idxmax()} ({vc.max()} próbek, {100*vc.max()/len(df):.1f}%)")


### 1.2 Statystyki opisowe cech

Wszystkie cechy wejściowe są **numeryczne** (parametry chemiczne). Różne kolumny mają **różne skale** (np. `density` ~1, `total sulfur dioxide` do setek) — stąd w modelu MLP stosujemy **StandardScaler** wewnątrz pipeline.


In [ ]:
desc = df[FEATURE_COLUMNS].describe().T
desc["brak_NaN"] = df[FEATURE_COLUMNS].isna().sum()
display(desc.round(3))


### 1.3 Macierz korelacji (Pearson)

Mapa korelacji pokazuje, które cechy są ze sobą powiązane. Wysoka |korelacja| (>0,7) sugeruje redundancję — w opcjonalnej selekcji cech można rozważyć usunięcie jednej z pary.


In [ ]:
corr = df[FEATURE_COLUMNS].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, annot=False)
ax.set_title("Macierz korelacji — 11 cech chemicznych")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Pary z |r| > 0.7 (informacyjnie)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pary = [(i, j, upper.loc[j, i]) for i in upper.index for j in upper.columns if pd.notna(upper.loc[j, i]) and abs(upper.loc[j, i]) > 0.7]
if pary:
    print("Pary cech z |korelacja| > 0,7:")
    for a, b, r in sorted(pary, key=lambda x: -abs(x[2])):
        print(f"  {a} — {b}: {r:.3f}")
else:
    print("Brak par z |korelacja| > 0,7.")


### 1.4 Zależność cech od jakości (przykłady)

Sprawdzamy, czy **alkohol** i **kwasowość lotna** różnią się między ocenami — to typowe predyktory jakości wina w literaturze.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x=TARGET_COLUMN, y="alcohol", ax=axes[0], palette="Blues")
axes[0].set_title("Alkohol vs ocena jakości")
axes[0].set_xlabel("quality")

sns.boxplot(data=df, x=TARGET_COLUMN, y="volatile acidity", ax=axes[1], palette="Oranges")
axes[1].set_title("Kwasowość lotna vs ocena jakości")
axes[1].set_xlabel("quality")

plt.tight_layout()
plt.show()


## 2. Eksperyment — modele i walidacja krzyżowa

Logika w `src/experiment.py`:
| Krok | Co robi |
|------|---------|
| Pipeline | `StandardScaler` → klasyfikator (bez przecieku: skaler tylko na treningu foldy) |
| Modele | **MLP** ×3 (główny), **Random Forest** ×3, **XGBoost** ×3 jeśli dostępny |
| CV | Stratyfikowana **5-fold**, `random_state=42` |
| Metryki | accuracy, balanced accuracy, F1-macro |
| Test | Najlepszy MLP (wg balanced accuracy w CV) na **20%** hold-out |

Ustaw `URUCHOM_OD_ZERA = False`, aby wczytać gotowe wyniki z `results/` (szybciej).  
Ustaw `True`, aby przeliczyć wszystko od nowa (jak `python run_experiment.py`).


In [ ]:
URUCHOM_OD_ZERA = False  # True = przelicz CV od zera (1–3 min); False = wczytaj results/

csv_sz = RESULTS_DIR / "wyniki_szczegolowe.csv"
csv_gr = RESULTS_DIR / "wyniki_agregaty_rodzin.csv"

if not URUCHOM_OD_ZERA and csv_sz.is_file() and csv_gr.is_file():
    print("Wczytuję zapisane wyniki z results/")
    szczegoly = pd.read_csv(csv_sz)
    grupy = pd.read_csv(csv_gr)
    # Macierz pomyłek — szybka ocena najlepszego MLP
    from src.experiment import wybierz_najlepszy_mlp, ocena_na_tescie
    X, y, nazwy = podziel_na_cechy_i_etykiete(df)
    wynik_test = ocena_na_tescie(X, y, nazwy, wybierz_najlepszy_mlp(szczegoly))
else:
    print("Uruchamiam pełny eksperyment (CV + test MLP)...")
    szczegoly, grupy, wynik_test = uruchom_pelny_eksperyment()

print("\n--- Najlepszy MLP (CV) ---")
from src.experiment import wybierz_najlepszy_mlp
print("Wariant:", wybierz_najlepszy_mlp(szczegoly))
print(f"Test accuracy: {wynik_test['accuracy']:.4f}")
print(f"Test balanced accuracy: {wynik_test['balanced_accuracy']:.4f}")
print(f"Test F1-macro: {wynik_test['f1_macro']:.4f}")


### 2.1 Tabela wyników — wszystkie warianty modeli

Kolumny `*_mean` i `*_std` to średnia i odchylenie standardowe z **5 foldów** walidacji krzyżowej.


In [ ]:
display(szczegoly.round(4))
display(grupy.round(4))


### 2.2 Wykres — porównanie accuracy (CV)

Słupki: średnia accuracy; **linie** (errorbar): odchylenie między foldami.


In [ ]:
df_plot = szczegoly.copy()
df_plot["_sort"] = df_plot["rodzina"].map({"MLP": 0, "RandomForest": 1, "XGBoost": 2}).fillna(9)
df_plot = df_plot.sort_values(["_sort", "model"])
kolory = {"MLP": "#E84855", "RandomForest": "#1B998B", "XGBoost": "#5C4D7D"}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metryka, std, tyt in zip(
    axes,
    ["accuracy_mean", "balanced_accuracy_mean", "f1_macro_mean"],
    ["accuracy_std", "balanced_accuracy_std", "f1_macro_std"],
    ["Accuracy", "Balanced accuracy", "F1-macro"],
):
    kol = [kolory.get(r, "#888") for r in df_plot["rodzina"]]
    ax.bar(range(len(df_plot)), df_plot[metryka], yerr=df_plot[std], capsize=3, color=kol, ecolor="gray")
    ax.set_xticks(range(len(df_plot)))
    ax.set_xticklabels(df_plot["model"], rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{tyt} (5-fold CV)")
    ax.set_ylabel("Średnia ± std")

plt.suptitle("Porównanie modeli — Wine Quality", y=1.02)
plt.tight_layout()
plt.show()


### 2.3 Agregaty po rodzinie modelu

Dla każdej rodziny (MLP / RF / XGB) pokazujemy **średnią** i **maksymalną** wartość metryki spośród 3 wariantów hiperparametrów.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(grupy))
w = 0.35
ax.bar(x - w/2, grupy["accuracy_srednia"], w, label="Śr. accuracy", color="#5C4D7D")
ax.bar(x + w/2, grupy["accuracy_max"], w, label="Maks. accuracy", color="#8F7EAF")
ax.set_xticks(x)
ax.set_xticklabels(grupy["rodzina"])
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Accuracy — średnia vs maksimum w rodzinie modelu")
plt.tight_layout()
plt.show()


## 3. Macierz pomyłek — najlepszy MLP na zbiorze testowym (20%)

Wiersze = **prawdziwa** ocena, kolumny = **predykcja**.  
Diagonalne komórki to trafne klasyfikacje. Błędy „obok” (np. 5↔6) są typowe przy podobnych klasach sensorycznych.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = wynik_test["confusion_matrix"]
etykiety = wynik_test["labels"]

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=etykiety).plot(
    ax=ax, cmap="Blues", colorbar=True
)
ax.set_title(f"Macierz pomyłek — {wynik_test['model']} (held-out test)")
plt.tight_layout()
plt.show()

cm_df = pd.DataFrame(cm, index=[f"prawda:{e}" for e in etykiety], columns=[f"pred:{e}" for e in etykiety])
display(cm_df)


## 4. Wnioski (do sprawozdania)

### Wyniki
- **Random Forest** (`rf_300`) osiąga zwykle **najlepszą accuracy** w CV (~0,69) i **F1-macro** (~0,40).
- **MLP** (`mlp_128_64_32` lub `mlp_128_64`) daje ~**0,61–0,62** accuracy w CV i ~**0,63** na teście.
- **Balanced accuracy** jest **niższa** (~0,35) niż accuracy — efekt **niezbalansowania** klas 5 i 6.

### Metodologia
- Pipeline ze **skalowaniem** jest poprawny (brak przecieku do CV).
- Porównanie **wielu wariantów** hiperparametrów w ramach MLP i RF jest zgodne z wytycznymi projektu 2.

### Ograniczenia i dalsze kroki
- Dokładne przewidywanie skrajnych ocen (3, 4, 8, 9) jest trudne przy małej liczbie próbek tych klas.
- Możliwe ulepszenia: **wagowanie klas**, grupowanie ocen (np. niska/średnia/wysoka), tuning MLP, **XGBoost** po `brew install libomp` (macOS).

### Powiązane pliki
- `run_experiment.py`, `streamlit run streamlit_app.py`, `results/wyniki_szczegolowe.csv`
